# German Tutor QLoRA Training

Fine-tunes Qwen2.5-3B-Instruct as a German tutor for Arabic speakers.

**Secrets (Add-ons → Secrets):**
- `HF_TOKEN` — Hugging Face write token (optional — if set, adapter auto-uploaded)
- `HF_REPO` — e.g. `mamonalburihy/german-tutor-lora`

If secrets are missing, training still runs. You can upload the adapter manually later.

In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path

WORKING = Path("/kaggle/working")
os.chdir(WORKING)

has_gpu = shutil.which('nvidia-smi') is not None
if has_gpu:
    print(f"GPU: {subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout}")
else:
    print("WARNING: No GPU found. Training will be VERY slow.")
print(f"Python: {sys.version}")

### 1. Load training data

In [ ]:
# Try mounted dataset first, fall back to kagglehub
DATA_DIR = Path("/kaggle/input/german-tutor-training-data")
if not DATA_DIR.exists():
    print("Dataset not mounted. Downloading via kagglehub...")
    import kagglehub
    DATA_DIR = Path(kagglehub.dataset_download("mamonalburihy/german-tutor-training-data"))

TRAIN_FILE = next(DATA_DIR.glob("*train*weighted*.jsonl"), None)
VAL_FILE = next(DATA_DIR.glob("*val*.jsonl"), None)
if not TRAIN_FILE or not VAL_FILE:
    print(f"Files in {DATA_DIR}: {list(DATA_DIR.iterdir())}")
    raise FileNotFoundError("Could not find training data files")

print(f"Train: {TRAIN_FILE.name} ({(TRAIN_FILE.stat().st_size / 1e6):.1f} MB)")
print(f"Val:   {VAL_FILE.name} ({(VAL_FILE.stat().st_size / 1e6):.1f} MB)")

### 2. Install dependencies

In [ ]:
# Install P100-compatible environment (no bitsandbytes — model loads in bf16)
# Kaggle's default torch 2.10.0+cu128 dropped sm_60 support, so use 2.5.0+cu124

# Step 1: Install P100-compatible PyTorch
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-deps",
    "torch==2.5.0+cu124",
    "--index-url", "https://download.pytorch.org/whl/cu124",
], check=True)

# Step 2: Install ML deps (no bitsandbytes — model loaded in bf16, no 4-bit quantization needed)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.45.2",
    "datasets==2.20.0",
    "accelerate==0.33.0",
    "peft==0.12.0",
    "sentencepiece==0.2.0",
    "safetensors>=0.4.3",
    "protobuf>=5.27.0",
], check=True)
print("All dependencies installed")


### 3. Run training

In [ ]:
cfg = json.loads((DATA_DIR / "qlora_config_german_tutor.json").read_text())
print(f"Config: {json.dumps(cfg, indent=2)}")

result = subprocess.run([
    sys.executable,
    str(DATA_DIR / "train_qlora_german_tutor.py"),
    "--train_file", str(TRAIN_FILE),
    "--val_file", str(VAL_FILE),
], capture_output=False, text=True)
result.check_returncode()
print("\nTraining completed successfully!")

### 4. Upload adapter to Hugging Face Hub (if secret set)

In [ ]:
ADAPTER_DIR = WORKING / cfg["output_dir"]

# Try to read secrets via multiple methods
HF_TOKEN = ""
HF_REPO = ""

# Method 1: Kaggle Secrets API (works when Run is clicked from Kaggle UI)
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    HF_TOKEN = s.get_secret("HF_TOKEN")
    HF_REPO = s.get_secret("HF_REPO")
except Exception:
    # Method 2: Environment variables
    HF_TOKEN = os.getenv("HF_TOKEN", "")
    HF_REPO = os.getenv("HF_REPO", "")

if not HF_TOKEN or not HF_REPO:
    print("=" * 60)
    print("HF_TOKEN or HF_REPO not set. To auto-upload:")
    print("  Add-ons \u2192 Secrets \u2192 Add HF_TOKEN and HF_REPO, then re-run")
    print()
    print("Manual upload later:")
    print(f"  Download from Kaggle working dir: {ADAPTER_DIR}")
    print(f"  Then: huggingface-cli upload <repo> . --repo-type model")
    print("=" * 60)
else:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)

    if not ADAPTER_DIR.exists():
        print(f"Adapter dir not found: {ADAPTER_DIR}")
        print(f"Contents of WORKING: {[p.name for p in WORKING.iterdir()]}")
        raise SystemExit(1)

    api = HfApi()
    try:
        api.create_repo(HF_REPO, private=True, exist_ok=True)
    except Exception as e:
        print(f"Repo may already exist: {e}")

    api.upload_folder(
        repo_id=HF_REPO,
        folder_path=str(ADAPTER_DIR),
        path_in_repo=".",
        commit_message=f"Train: {cfg['train_file']}, {cfg['num_train_epochs']} epochs",
        delete_patterns="*",
    )
    print(f"\nAdapter uploaded to https://huggingface.co/{HF_REPO}")

print(f"\nDownload back on your server:")
print(f"  huggingface-cli download {HF_REPO or '<your-repo>'} --local-dir training/qwen25_german_tutor_qlora")